## 📘 Apache Airflow – Deep Theory & Interview Preparation Notebook

#### 📌 1. What is Apache Airflow? (Definition)

Apache Airflow is an open-source workflow orchestration platform used to programmatically author, schedule, monitor, and manage data pipelines.

Airflow treats workflows as code, written primarily in Python, enabling dynamic pipeline generation, version control, testing, and reusability.

#### Key Definition Points (Interview-ready):
- Airflow is not a data processing tool
- It orchestrates tasks, it does not execute heavy transformations itself
- Pipelines are defined as DAGs (Directed Acyclic Graphs)

#### 📌 2. What Problem Does Airflow Solve?
##### ❌ Problems WITHOUT Airflow

Before Airflow, teams faced:

- Cron jobs with no dependency tracking
- Manual retries and monitoring
- No centralized logging
- Hard-to-debug pipeline failures
- Poor scalability
Example (Cron Hell):

In [ ]:
1 AM → run job A
2 AM → run job B (depends on A)
3 AM → run job C (depends on B)
❌ If Job A fails → everything breaks silently

#### ✅ Problems Airflow Solves

| Problem           | How Airflow Solves It           |
| ----------------- | ------------------------------- |
| Task dependencies | DAG-based dependency management |
| Scheduling        | Built-in scheduler              |
| Failure handling  | Retries, alerts, SLAs           |
| Monitoring        | Web UI                          |
| Scaling           | Distributed execution           |
| Version control   | Pipelines as code               |


#### 📌 3. Why Airflow? (Why Not Cron / Scripts)

##### Why companies use Airflow:

- Airbnb (creator)
- Uber, Netflix, Spotify, Stripe

##### Key Reasons:

- Python-based → flexible logic
- Dynamic DAGs
- Strong community
- Cloud-native (AWS, GCP, Azure)
- Rich operator ecosystem

📌 Airflow is the industry standard orchestrator for Data Engineering

#### 📌 4. Core Airflow Concepts (VERY IMPORTANT)
##### 🔹 4.1 DAG (Directed Acyclic Graph)
##### Definition:

A DAG represents a workflow where:
- Nodes = Tasks
- Edges = Dependencies
- No cycles allowed

##### 📌 DAG answers:
- What to run
- In what order
- On what schedule

#### ✅ DAG Properties

| Property            | Meaning                   |
| ------------------- | ------------------------- |
| `dag_id`            | Unique DAG name           |
| `start_date`        | When DAG starts           |
| `schedule_interval` | When DAG runs             |
| `catchup`           | Backfill past runs or not |


In [ ]:
from airflow import DAG
from datetime import datetime

with DAG(
    dag_id="example_dag",
    start_date=datetime(2024, 1, 1),
    schedule_interval="@daily",
    catchup=False
) as dag:
    pass
#📌 DAG file is parsed, not executed, by the scheduler

#### 🔹 4.2 Task
##### Definition:
A Task is a single unit of work in Airflow.

Examples:
- Extract data
- Transform data
- Load data
- Run SQL
- Trigger Spark job

###### Task Types:

- PythonOperator
- BashOperator
- SQL operators
- Custom Operators

In [ ]:
from airflow.operators.python import PythonOperator

def print_hello():
    print("Hello Airflow")

task_1 = PythonOperator(
    task_id="hello_task",
    python_callable=print_hello
)
#📌 Tasks are idempotent (safe to retry)

#### 🔹 4.3 Operator
#### Definition:

An Operator defines how a task should be executed.

##### 📌 Think of Operator as:

“Template for a task”

#### Operator Categories (Very Important)

Category           | Examples                     |
| ------------------ | ---------------------------- |
| Action Operators   | BashOperator, PythonOperator |
| Transfer Operators | S3ToRedshiftOperator         |
| Sensor Operators   | FileSensor                   |
| Branch Operators   | BranchPythonOperator         |


#### 🔹 4.4 Sensor (Interview Favorite)
##### Definition:
A Sensor is a special operator that waits for a condition.

Examples:
- Wait for file
- Wait for table
- Wait for API response

In [ ]:
from airflow.sensors.filesystem import FileSensor

wait_for_file = FileSensor(
    task_id="wait_for_file",
    filepath="/data/input.csv",
    poke_interval=30,
    timeout=300
)

#📌 Sensors block worker slots unless using reschedule mode


#### 📌 5. Airflow Architecture (VERY IMPORTANT)

In [ ]:
          +-------------------+
          |     Web Server    |
          +-------------------+
                    |
+-----------+    +----------------+    +------------+
| Scheduler | -> | Metadata DB    | <- | Executors  |
+-----------+    +----------------+    +------------+


#### 🔹 Scheduler
- Parses DAG files
- Schedules task instances
- Decides when tasks should run

#### 🔹 Executor

- Executes tasks
- Types:
    - SequentialExecutor
    - LocalExecutor
    - CeleryExecutor
    - KubernetesExecutor

📌 Executor decides scalability

#### 🔹 Metadata Database

##### Stores:
- DAG runs
- Task states
- Logs metadata

##### Supported:
- PostgreSQL
- MySQL

#### 🔹 Workers
- Actually run the tasks
- Controlled by executor

In [ ]:
📌 6. Executors Explained (Interview Critical)
Executor	Use Case
SequentialExecutor	Local testing
LocalExecutor	Single machine parallel
CeleryExecutor	Distributed (Prod)
KubernetesExecutor	Cloud-native

📌 CeleryExecutor = Most common in production

📌 7. Scheduling & DAG Runs
DAG Run vs Task Instance
Term	Meaning
DAG Run	One execution of DAG
Task Instance	One task in one DAG run

📌 DAG can have multiple active DAG runs

Schedule Interval Examples
schedule_interval="@daily"
schedule_interval="0 2 * * *"
schedule_interval=None  # Manual

📌 8. Dependency Management
Task Dependencies
task_1 >> task_2 >> task_3


OR

task_1.set_downstream(task_2)


📌 Airflow builds execution graph from dependencies

📌 9. XCom (Inter-task Communication)
What is XCom?

XCom allows tasks to exchange small pieces of data.

📌 Not for large datasets

XCom Example
def push_value(**context):
    context['ti'].xcom_push(key='count', value=10)

def pull_value(**context):
    value = context['ti'].xcom_pull(key='count')
    print(value)


📌 Stored in metadata DB

📌 10. Hooks & Connections
Connection

Stores credentials securely

Managed via UI or environment variables

Hook

Interface to external systems

Uses connection internally

Examples:

PostgresHook

S3Hook

MySqlHook

📌 11. Plugins in Airflow
What are Plugins?

Plugins allow:

Custom Operators

Custom Hooks

UI extensions

📌 Used when built-in operators are insufficient

Plugin Structure
plugins/
 ├── custom_operator.py
 ├── custom_hook.py
 └── airflow_plugin.py


📌 Plugins loaded at scheduler startup

📌 12. Error Handling & Retries
Retry Configuration
default_args = {
    "retries": 3,
    "retry_delay": timedelta(minutes=5)
}


📌 Airflow retries only failed tasks, not entire DAG

📌 13. When to Use Airflow (IMPORTANT)
Use Airflow When:

Complex dependencies

Scheduled pipelines

Need monitoring

Multi-step workflows

Do NOT Use Airflow When:

Real-time streaming

Low-latency execution

Simple scripts

📌 14. Airflow vs Other Tools (Interview)
Tool	Purpose
Airflow	Orchestration
Spark	Processing
DBT	Transformations
Kafka	Streaming

📌 Airflow coordinates, others process

📌 15. Common Interview Questions (Quick)

Q1: Is Airflow a data processing tool?
👉 No, it orchestrates tasks.

Q2: Why DAG must be acyclic?
👉 To avoid infinite loops.

Q3: Where are XComs stored?
👉 Metadata DB.

Q4: Which executor for production?
👉 Celery / Kubernetes.

📌 16. Final Summary

Airflow is a workflow orchestrator

DAGs define pipelines

Operators define tasks

Scheduler + Executor run workflows

Highly scalable and production-ready